In [29]:
from pathlib import Path

from financial_analysis_agent.core.config import settings
from financial_analysis_agent.data.download_sec import download_company_facts

# Project root: adapter if necessary
PROJECT_ROOT = Path.cwd().parent

# Apple SEC ID
APPLE_CIK = "320193"

# Save directory
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "companyfacts"

# Downloading
json_path = download_company_facts(
    cik=APPLE_CIK,
    user_agent=settings.sec_user_agent,
    output_dir=RAW_DATA_DIR
)

print("File saved :", json_path)

File saved : C:\Users\Aboubacar\Documents\data_science_ai\Financial-analysis-agent\data\raw\companyfacts\CIK0000320193_companyfacts.json


In [2]:
import  json

# Read json file in Python object
with json_path.open("r", encoding="utf-8") as file:
    company_data = json.load(file)

print("CIK :", company_data["cik"])
print("Company :", company_data["entityName"])
print("Taxonomies :", list(company_data["facts"].keys()))

CIK : 320193
Company : Apple Inc.
Taxonomies : ['dei', 'us-gaap']


In [3]:
# Get US-GAAP financial variables
us_gaap_facts = company_data["facts"]["us-gaap"]

# Number of variables available
print("Number of US-Gaap variables:", len(us_gaap_facts))

# Show the 15 first variables
for variable in list(us_gaap_facts.keys())[:5]:
    print(variable)

Number of US-Gaap variables: 503
AccountsPayable
AccountsPayableCurrent
AccountsReceivableNetCurrent
AccruedIncomeTaxesCurrent
AccruedIncomeTaxesNoncurrent


In [5]:
# Get assets variable data
assets = us_gaap_facts["Assets"]

print("Label:", assets["label"])
print("Description:", assets["description"])
print("Units:", list(assets["units"].keys()))

Label: Assets
Description: Sum of the carrying amounts as of the balance sheet date of all assets that are recognized. Assets are probable future economic benefits obtained or controlled by an entity as a result of past transactions or events.
Units: ['USD']


In [6]:
# Get the observation total of assets
assets_observations = assets["units"]["USD"]

# Numbers of observations
print("Number of observations:", len(assets_observations))

# Show the first observation
print(assets_observations[0])

Number of observations: 146
{'end': '2008-09-27', 'val': 39572000000, 'accn': '0001193125-09-153165', 'fy': 2009, 'fp': 'Q3', 'form': '10-Q', 'filed': '2009-07-22'}


In [8]:
import  numpy as np
import  pandas as pd
import  matplotlib.pyplot as plt

In [8]:
df_assets = pd.DataFrame(assets_observations)
display(df_assets.head(3))

,end,val,accn,fy,fp,form,filed,frame
0,2008-09-27,39572000000,0001193125-09-153165,2009.0,Q3,10-Q,2009-07-22,NaN
1,2008-09-27,39572000000,0001193125-09-214859,2009.0,FY,10-K,2009-10-27,NaN
2,2008-09-27,36171000000,0001193125-10-012091,2009.0,FY,10-K/A,2010-01-25,NaN


In [9]:
# Get revenue variable data
revenue = us_gaap_facts["Revenues"]

revenue_observations = revenue["units"]["USD"]
df_revenue = pd.DataFrame(revenue_observations)
display(df_revenue.head(3))

,start,end,val,accn,fy,fp,form,filed,frame
0,2015-09-27,2016-09-24,215639000000,0000320193-18-000145,2018,FY,10-K,2018-11-05,CY2016
1,2016-09-25,2016-12-31,78351000000,0000320193-18-000145,2018,FY,10-K,2018-11-05,CY2016Q4
2,2017-01-01,2017-04-01,52896000000,0000320193-18-000145,2018,FY,10-K,2018-11-05,CY2017Q1


In [18]:
df_revenue.shape

(11, 9)

In [4]:
# Build a list of us gaap facts variable needed for financial tool
variable_list = {
    "total_assets": "Assets",
    "revenue": "RevenueFromContractWithCustomerExcludingAssessedTax",
    "gross_profit": "GrossProfit",
    "net_income": "NetIncomeLoss",
    "shareholders_equity": "StockholdersEquity",
}

In [20]:
# Check variable available in XBRL data
for variable in variable_list:
    if variable in us_gaap_facts:
        observations = us_gaap_facts[variable]["units"].get("USD", [])
        print(f"{variable}: AVAILABLE ({len(observations)} observations)")

    else:
        print(f"{variable}: NOT AVAILABLE")


Assets: AVAILABLE (146 observations)
RevenueFromContractWithCustomerExcludingAssessedTax: AVAILABLE (117 observations)
GrossProfit: AVAILABLE (338 observations)
NetIncomeLoss: AVAILABLE (338 observations)
StockholdersEquity: AVAILABLE (264 observations)


In [7]:
# Check the number of observations for each variable between 2020 and 2025
START_YEAR = 2020
END_YEAR = 2026

for variable, concept in variable_list.items():
    observations = us_gaap_facts[concept]["units"]["USD"]

    annual_observations = [
        obs for obs in observations
        if obs.get("form") == "10-K"
           and START_YEAR <= int(obs["end"][:4]) <= END_YEAR
    ]

    years = sorted({
        obs["end"][:4]
        for obs in annual_observations
    })

    print(f"\n{variable}: {len(annual_observations)} observations")
    print(f"Years: {years}")


total_assets: 11 observations
Years: ['2020', '2021', '2022', '2023', '2024', '2025']

revenue: 18 observations
Years: ['2020', '2021', '2022', '2023', '2024', '2025']

gross_profit: 18 observations
Years: ['2020', '2021', '2022', '2023', '2024', '2025']

net_income: 18 observations
Years: ['2020', '2021', '2022', '2023', '2024', '2025']

shareholders_equity: 18 observations
Years: ['2020', '2021', '2022', '2023', '2024', '2025']


In [12]:
# Number of filings by year for each variable
records = []
for variable, concept in variable_list.items():
    observations = us_gaap_facts[concept]["units"]["USD"]

    for obs in observations:
        if (
                obs.get("form") == "10-K"
                and 2020 <= int(obs["end"][:4]) <= 2025
        ):
            records.append({
                "variable": variable,
                "start": obs.get("start"),
                "end": obs.get("end"),
                "form": obs.get("form"),
                "fy": obs.get("fy"),
                "fp": obs.get("fp"),
                "filed": obs.get("filed"),
                "accn": obs.get("accn"),
                "value": obs.get("val"),
            })

df = pd.DataFrame(records)
df["year"] = df["end"].str[:4].astype(int)

summary = (
    df.groupby(["variable", "year"])
    .size()
    .unstack(fill_value=0)
)

display(summary)

year,2020,2021,2022,2023,2024,2025
variable,,,,,,
gross_profit,6,3,3,3,2,1
net_income,6,3,3,3,2,1
revenue,6,3,3,3,2,1
shareholders_equity,4,4,4,3,2,1
total_assets,2,2,2,2,2,1


In [15]:
# Gross profit filings for 2023 analysis
gross_profit_23 = df[
    (df["variable"] == "gross_profit")
    & (df["year"] == 2023)
]

display(
    gross_profit_23[
        ["start", "end", "fy", "fp", "filed", "accn", "value"]
    ].sort_values("filed")
)

,start,end,fy,fp,filed,accn,value
41,2022-09-25,2023-09-30,2023,FY,2023-11-03,0000320193-23-000106,169148000000
42,2022-09-25,2023-09-30,2024,FY,2024-11-01,0000320193-24-000123,169148000000
43,2022-09-25,2023-09-30,2025,FY,2025-10-31,0000320193-25-000079,169148000000


In [21]:
# Build array for each variable and each account year
records = []

for variable, concept in variable_list.items():
    observations = us_gaap_facts[concept]["units"]["USD"]

    for obs in observations:
        fy = obs.get("fy")
        fp = obs.get("fp")
        form = obs.get("form")
        accn = obs.get("accn")

        # Search for candidate's original annual filing
        if (
            form == "10-K"
            and fp == ("FY")
            and isinstance(fy, int)
            and 2020 <= fy <= 2025
            and accn[11:13] == str(fy)[-2:]
        ):
            records.append({
                "variable": variable,
                "fiscal_year": fy,
                "start": obs.get("start"),
                "end": obs["end"],
                "value": obs["val"],
                "accn": accn,
                "filed": obs.get("filed"),
            })

df_annual = pd.DataFrame(records)
display(df_annual.head(15))



,variable,fiscal_year,start,end,value,accn,filed
0,total_assets,2020,NaN,2019-09-28,338516000000,0000320193-20-000096,2020-10-30
1,total_assets,2020,NaN,2020-09-26,323888000000,0000320193-20-000096,2020-10-30
2,total_assets,2021,NaN,2020-09-26,323888000000,0000320193-21-000105,2021-10-29
3,total_assets,2021,NaN,2021-09-25,351002000000,0000320193-21-000105,2021-10-29
4,total_assets,2022,NaN,2021-09-25,351002000000,0000320193-22-000108,2022-10-28
5,total_assets,2022,NaN,2022-09-24,352755000000,0000320193-22-000108,2022-10-28
6,total_assets,2023,NaN,2022-09-24,352755000000,0000320193-23-000106,2023-11-03
7,total_assets,2023,NaN,2023-09-30,352583000000,0000320193-23-000106,2023-11-03
8,total_assets,2024,NaN,2023-09-30,352583000000,0000320193-24-000123,2024-11-01
9,total_assets,2024,NaN,2024-09-28,364980000000,0000320193-24-000123,2024-11-01
